In [36]:
import sys
import os
import psycopg2
from dotenv import load_dotenv
import QuantLib as ql


sys.path.append(os.path.abspath(".."))
from utils.black_scholes import get_implied_vol

load_dotenv()

url = os.getenv("DATABASE_URL")

In [23]:
conn = psycopg2.connect(url, sslmode="require")
cur = conn.cursor()

In [24]:
postfix_list = [
    "CD6", #04-15
]

ticker = "SR"
min_strike = 270
max_strike = 370
strike_step = 10

strike = min_strike
tickers = []
while strike < max_strike:
    for postfix in postfix_list:
        option_ticker = ticker + str(strike) + postfix
        tickers.append(option_ticker)
    strike += strike_step
print(tickers)

['SR270CD6', 'SR280CD6', 'SR290CD6', 'SR300CD6', 'SR310CD6', 'SR320CD6', 'SR330CD6', 'SR340CD6', 'SR350CD6', 'SR360CD6']


In [25]:
query = """
SELECT DISTINCT ON (ticker)
    ticker,
    bids,
    asks,
    timestamp
FROM orderbooks
WHERE ticker = ANY(%s)
ORDER BY ticker, timestamp DESC;
"""

cur.execute(query, (tickers,))
rows = cur.fetchall()

In [26]:
rows

[('SR270CD6',
  [{'price': 10.15, 'quantity': 10}],
  [],
  datetime.datetime(2026, 4, 15, 15, 54, 57, tzinfo=datetime.timezone.utc)),
 ('SR280CD6',
  [{'price': 10.15, 'quantity': 10}],
  [],
  datetime.datetime(2026, 4, 15, 15, 54, 57, tzinfo=datetime.timezone.utc)),
 ('SR290CD6',
  [{'price': 10.15, 'quantity': 10}],
  [],
  datetime.datetime(2026, 4, 15, 15, 54, 57, tzinfo=datetime.timezone.utc)),
 ('SR300CD6',
  [{'price': 0.02, 'quantity': 110}],
  [],
  datetime.datetime(2026, 4, 15, 15, 59, 13, tzinfo=datetime.timezone.utc)),
 ('SR310CD6',
  [{'price': 10.0, 'quantity': 2}],
  [{'price': 11.0, 'quantity': 3000}],
  datetime.datetime(2026, 4, 15, 16, 0, 30, tzinfo=datetime.timezone.utc)),
 ('SR320CD6',
  [{'price': 0.65, 'quantity': 1000},
   {'price': 0.1, 'quantity': 50000},
   {'price': 0.01, 'quantity': 14}],
  [{'price': 0.99, 'quantity': 1510}, {'price': 1.32, 'quantity': 400}],
  datetime.datetime(2026, 4, 15, 16, 0, 30, tzinfo=datetime.timezone.utc)),
 ('SR330CD6',
  [],

In [32]:
def compute_mid_price(bids, asks):
    best_bid = None
    best_ask = None
    
    if bids:
        best_bid = max(bids, key=lambda x: x["price"])["price"]
    if asks:
        best_ask = min(asks, key=lambda x: x["price"])["price"]

    if best_bid is not None and best_ask is not None:
        return (best_bid + best_ask) / 2
    return best_bid or best_ask

In [35]:
data = rows 
result = []

for ticker, bids, asks in data:
    mid = compute_mid_price(bids, asks)

    result.append({
        "ticker": ticker,
        "mid": mid,
    })
result

[{'ticker': 'SR270CD6', 'mid': 10.15},
 {'ticker': 'SR280CD6', 'mid': 10.15},
 {'ticker': 'SR290CD6', 'mid': 10.15},
 {'ticker': 'SR300CD6', 'mid': 0.02},
 {'ticker': 'SR310CD6', 'mid': 10.5},
 {'ticker': 'SR320CD6', 'mid': 0.8200000000000001},
 {'ticker': 'SR330CD6', 'mid': 0.05},
 {'ticker': 'SR340CD6', 'mid': 0.05},
 {'ticker': 'SR350CD6', 'mid': 0.54},
 {'ticker': 'SR360CD6', 'mid': 0.54}]